# 🛡️ Bot Guardián (WFM Real-Time Sentinel) v1.0 | LOB COACH
**Guía interactiva de ejecución en Google Colab**
- **Responsable Técnico:** Juan Esteban Serna Gutiérrez (Área WFM)

### Paso 1: Instalar dependencias e inicializar Chromium Headless
Ejecuta esta celda para instalar `playwright`, `fastapi`, `uvicorn`, `aiohttp`, `python-dotenv` y las librerías del sistema Linux que requiere Chromium en Google Colab.

In [ ]:
!pip install -q fastapi==0.115.0 uvicorn[standard]==0.30.6 playwright==1.47.0 aiohttp==3.10.5 python-dotenv==1.0.1 nest_asyncio
!playwright install chromium
!playwright install-deps chromium

### Paso 2: Obtener el código fuente (Clonar desde Git o Subir archivos)
- **Opción A:** Si ya subiste el proyecto a GitHub, descomenta la línea `!git clone ...`.
- **Opción B:** Sube los archivos `config.py`, `notifier.py`, `bot_guardian.py` y `main.py` al panel izquierdo de archivos (📁) de Google Colab.

In [ ]:
# Si lo tienes en un repositorio público o con token, descomenta:
# !git clone https://github.com/Juaness601/NOMBRE_DE_TU_REPO.git
# %cd NOMBRE_DE_TU_REPO

!ls -la

### Paso 3: Configurar Variables de Entorno (Credenciales, Webhook y Umbrales)
Puedes usar el panel de **Secrets (🔑)** de Google Colab (en la barra lateral izquierda) o definirlas directamente aquí para tu sesión temporal.

In [ ]:
import os

# --- URLs y Credenciales WFM ---
os.environ["WFM_DASHBOARD_URL"] = "https://wfm-plataforma.ejemplo.com/realtime/coach"
os.environ["WFM_LOGIN_URL"] = "https://wfm-plataforma.ejemplo.com/login"
os.environ["WFM_USERNAME"] = "tu_usuario_wfm"
os.environ["WFM_PASSWORD"] = "tu_password_wfm"

# --- Webhook Oficial de Google Chat ---
os.environ["GOOGLE_CHAT_WEBHOOK_URL"] = "https://chat.googleapis.com/v1/spaces/XXXX/messages?key=YYYY&token=ZZZZ"

# --- Intervalo y Umbrales WFM (en segundos) ---
os.environ["POLL_INTERVAL_SECONDS"] = "15"
os.environ["TIMEZONE_OFFSET_HOURS"] = "-5"
os.environ["THRESHOLD_LONG_CALL"] = "900"    # 15 min
os.environ["THRESHOLD_OVER_BREAK"] = "900"   # 15 min
os.environ["THRESHOLD_OVER_LUNCH"] = "2700"  # 45 min

# --- Selectores CSS del DOM ---
os.environ["SELECTOR_AGENT_ROWS"] = "table#agents-grid tbody tr, .agent-row"
os.environ["SELECTOR_AGENT_NAME"] = ".agent-name, td:nth-child(1)"
os.environ["SELECTOR_AGENT_STATE"] = ".agent-state, td:nth-child(2)"
os.environ["SELECTOR_AGENT_TIME"] = ".agent-time, td:nth-child(3)"

print("✅ Variables de entorno cargadas en la sesión de Colab.")

### Paso 4 (Prueba Rápida): Probar conexión con el Webhook de Google Chat
Antes de lanzar el monitoreo completo, verifica que tu `GOOGLE_CHAT_WEBHOOK_URL` recibe correctamente la tarjeta **Card v2**.

In [ ]:
import importlib
import config
import notifier

importlib.reload(config)
importlib.reload(notifier)

exito = await notifier.send_google_chat_alert(
    agent_name="Agente de Prueba (Colab)",
    event_type="Long Call",
    current_state="En Llamada",
    elapsed_seconds=945,
    threshold_seconds=900
)
print("Resultado del envío a Google Chat:", "✅ Exitoso" if exito else "❌ Falló (revisa tu Webhook URL)")

### Paso 5 (Opción A - Directa): Ejecutar el Monitoreo en Tiempo Real dentro de Colab
Ejecuta esta celda para iniciar el centinela. Para detenerlo limpiamente en cualquier momento, presiona el botón **Stop (⏹️)** de la celda.

In [ ]:
import asyncio
import logging
import importlib
import config
import bot_guardian

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
importlib.reload(config)
importlib.reload(bot_guardian)

sentinel = bot_guardian.BotGuardian()

try:
    await sentinel.run_monitoring_loop()
except (KeyboardInterrupt, asyncio.CancelledError):
    print("⏹️ Deteniendo Bot Guardián bajo demanda...")
    await sentinel.stop()

### Paso 5 (Opción B - Servidor FastAPI): Levantar la API `main.py` en segundo plano y probar `/start` y `/stop`
Si deseas probar exactamente el comportamiento HTTP de **Google Cloud Run** (`GET /`, `POST /start`, `POST /stop`) dentro de Colab:

In [ ]:
import subprocess
import time
import requests

# 1. Iniciar el servidor FastAPI (Uvicorn) en segundo plano en el puerto 8080
server_proc = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080"])
time.sleep(3)

# 2. Consultar Healthcheck (GET /)
print("GET / ->", requests.get("http://127.0.0.1:8080/").json())

# 3. Iniciar Monitoreo (POST /start)
print("POST /start ->", requests.post("http://127.0.0.1:8080/start").json())
time.sleep(2)
print("GET / (activo) ->", requests.get("http://127.0.0.1:8080/").json())

In [ ]:
# 4. Detener Monitoreo limpiamente (POST /stop) y apagar Uvicorn
import requests
print("POST /stop ->", requests.post("http://127.0.0.1:8080/stop").json())
server_proc.terminate()